# 🏦 Monte Carlo Portfolio Risk Simulation — Capstone

This notebook implements a **Monte Carlo simulation** to model the one-year risk profile of a 5-asset equity portfolio.

## What it does
- Simulates **50,000 possible futures** for the portfolio over 252 trading days
- Models realistic **correlated asset returns** using Cholesky decomposition
- Computes two industry-standard risk metrics: **Value at Risk (VaR)** and **Conditional VaR (CVaR)**

## Phases at a glance
| Phase | Description |
|-------|-------------|
| 1 | Setup — portfolio parameters, returns, volatilities, correlation matrix |
| 2 | Cholesky decomposition — turn correlation into a "mixing recipe" |
| 3 | Generate independent random shocks |
| 4 | Apply correlation via matrix multiply |
| 5 | Simulate daily returns and compound them |
| 6 | Compute risk metrics (VaR, CVaR) |
| 7 | Run, time, and print results |

> **NumPy concepts used:** broadcasting, `np.random.randn`, `np.cumprod`, `np.percentile`, boolean masking, `np.linalg.cholesky`, matrix multiplication (`@`)


## Phase 1 — Setup Parameters

### Portfolio assumptions
- **$1,000,000** invested equally across **5 assets** (20% each)
- **50,000 simulations** — enough for stable percentile estimates at the 5% tail
- **252 trading days** — the standard US market year

### Annual → Daily conversion
We model *daily* returns because the simulation steps forward one day at a time.

| Quantity | Formula | Intuition |
|----------|---------|-----------|
| Daily mean return | `μ_daily = μ_annual / 252` | Simple scaling (continuously compounded approx.) |
| Daily volatility | `σ_daily = σ_annual / √252` | Variance scales linearly with time → std dev scales with √time |

### Correlation matrix
A symmetric 5×5 matrix where entry `[i, j]` is the **Pearson correlation** between assets i and j.
- Diagonal is always 1.0 (an asset is perfectly correlated with itself)
- Off-diagonal values between -1 and 1 — here mostly positive (0.1–0.6), typical for stocks in the same market

> A valid correlation matrix must be **positive semi-definite** (all eigenvalues ≥ 0).  
> This is required for Cholesky decomposition in Phase 2 to work.


In [7]:
import numpy as np
import time

# PHASE 1: Setup Parameters
np.random.seed(42)  # For reproducible results

# Portfolio settings
INITIAL_PORTFOLIO = 1_000_000
WEIGHTS = np.array([0.2, 0.2, 0.2, 0.2, 0.2])  # Equal weight (20% each)
N_ASSETS = len(WEIGHTS)
N_DAYS = 252           # Trading days in a year
N_SIMULATIONS = 50_000 # Number of possible futures

# Annual expected returns and volatilities (example values)
annual_returns = np.array([0.08, 0.12, 0.06, 0.10, 0.05])   # 8%, 12%, etc.
annual_volatility = np.array([0.20, 0.25, 0.15, 0.22, 0.18]) # 20%, 25%, etc.

# Convert to daily values
daily_returns = annual_returns / N_DAYS
daily_vol = annual_volatility / np.sqrt(N_DAYS)

# Correlation matrix (5x5)
corr_matrix = np.array([
    [1.00, 0.60, 0.30, 0.40, 0.20],
    [0.60, 1.00, 0.20, 0.50, 0.10],
    [0.30, 0.20, 1.00, 0.30, 0.50],
    [0.40, 0.50, 0.30, 1.00, 0.40],
    [0.20, 0.10, 0.50, 0.40, 1.00]
])

## Phase 2 — Cholesky Decomposition

### The problem
`np.random.randn(...)` generates **independent** (uncorrelated) random numbers.  
But real assets move together — when tech stocks fall, they often fall *together*.  
We need to *inject* the correlations from our `corr_matrix` into the random shocks.

### The solution: Cholesky decomposition
For any positive-definite matrix **Σ**, Cholesky finds a lower-triangular matrix **L** such that:

```
Σ = L @ L.T
```

**L** acts as a "mixing matrix". If **Z** is a vector of independent standard normals, then:

```
X = L @ Z   →   Cov(X) = L @ I @ L.T = Σ
```

So multiplying independent normals by **L** produces correlated normals with exactly the correlation structure we want.

### Why lower-triangular?
- Unique factorisation (Cholesky is deterministic for positive-definite matrices)
- Efficient: only N(N+1)/2 non-zero entries instead of N²
- Direct interpretation: asset 0's shocks are purely its own; asset 1 is a mix of asset 0 and its own noise; etc.

```python
L = np.linalg.cholesky(corr_matrix)
# L.shape == (5, 5), lower triangular
```


In [8]:
# PHASE 2: Cholesky Decomposition (Correlation Recipe)
L = np.linalg.cholesky(corr_matrix)  # Lower triangular mixing matrix

## Phases 3–5 — Monte Carlo Simulation Core

This single function runs the entire simulation. Let's walk through each step.

### Step 1 — Generate independent shocks
```python
uncorrelated = np.random.randn(N_DAYS, N_SIMULATIONS, N_ASSETS)
# Shape: (252, 50_000, 5)
```
- **252** days × **50,000** simulations × **5** assets = 63 million random numbers
- Each is an independent standard normal Z ~ N(0, 1)
- Axis 0 = time, Axis 1 = simulation paths, Axis 2 = assets

### Step 2 — Apply correlation
```python
correlated = uncorrelated @ L.T
# Shape: (252, 50_000, 5)
```
- Matrix multiply each `(5,)` shock vector by `L.T`
- Broadcasting handles the (252, 50_000) prefix dimensions automatically
- After this step, the 5 asset shocks at each timestep have the desired correlation structure

### Step 3 — Convert to daily returns (Geometric Brownian Motion)
```python
daily_shocks = daily_vol * correlated + daily_returns
```
- **Scale:** multiply by `daily_vol` to get the right volatility (std dev) per asset
- **Shift:** add `daily_returns` (the drift / expected return)
- Broadcasting: `daily_vol` is shape `(5,)` → broadcasts over `(252, 50_000, 5)`

This models each asset as: `r_t = μ_daily + σ_daily * Z_t` (discrete-time GBM)

### Step 4 — Compound returns over time
```python
cumulative_growth = np.cumprod(1 + daily_shocks, axis=0)
final_growth = cumulative_growth[-1]   # Shape: (50_000, 5)
```
- `1 + r_t` is the daily growth factor
- `np.cumprod(..., axis=0)` multiplies growth factors day by day along the time axis
- `[-1]` selects the last day — the final accumulated growth after 252 days

### Step 5 — Compute portfolio value
```python
initial_per_asset = INITIAL_PORTFOLIO * WEIGHTS  # $200,000 per asset
final_values = (initial_per_asset * final_growth).sum(axis=1)  # Shape: (50_000,)
```
- Multiply each asset's initial capital by its growth factor
- Sum across the 5 assets to get total portfolio value per simulation
- Result: 50,000 possible end-of-year portfolio values


In [9]:
# PHASE 3-5: Monte Carlo Simulation Function
def run_simulation():
    """Run 50,000 portfolio simulations and return final values."""
    # Generate independent standard normal random numbers
    uncorrelated = np.random.randn(N_DAYS, N_SIMULATIONS, N_ASSETS)
    
    # Apply correlation via Cholesky factor
    correlated = uncorrelated @ L.T
    
    # Convert to daily returns: return = mean + volatility * shock
    daily_shocks = daily_vol * correlated + daily_returns
    
    # Cumulative growth: product of (1 + return) over all days
    cumulative_growth = np.cumprod(1 + daily_shocks, axis=0)
    
    # Final growth factor for each asset (last day)
    final_growth = cumulative_growth[-1]  # Shape: (50000, 5)
    
    # Initial capital per asset
    initial_per_asset = INITIAL_PORTFOLIO * WEIGHTS  # Shape: (5,)
    
    # Final portfolio values: sum across assets
    final_values = (initial_per_asset * final_growth).sum(axis=1)  # Shape: (50000,)
    
    return final_values

## Phase 7 — Run and Time the Simulation

`time.perf_counter()` gives the highest-resolution clock available on the system — ideal for benchmarking.

The simulation must complete in **under 10 seconds** to pass the performance assertion.  
At 50,000 simulations × 252 days × 5 assets = **63 million** operations per step, efficient NumPy vectorization (no Python loops) makes this feasible.


In [10]:
# PHASE 7: Run and Time
print("Running Monte Carlo simulation...")
start_time = time.perf_counter()
final_portfolio = run_simulation()
elapsed = time.perf_counter() - start_time


Running Monte Carlo simulation...


## Phase 6 — Risk Metrics

### P&L (Profit and Loss)
```python
pnl = final_portfolio - INITIAL_PORTFOLIO
```
Simple: how much did we gain or lose compared to the starting $1,000,000?

### Value at Risk (VaR) at 95% confidence
```python
var_95 = np.percentile(pnl, 5)   # 5th percentile of P&L
```
**Interpretation:** "With 95% confidence, we will not lose more than `|var_95|` in one year."  
Equivalently: there is a 5% chance of losing *more* than this amount.

> VaR is the industry standard for regulatory reporting (Basel III) and internal risk limits.

### Conditional VaR (CVaR / Expected Shortfall)
```python
tail_mask = pnl <= var_95          # Boolean mask: worst 5% of outcomes
cvar = pnl[tail_mask].mean()       # Average loss in that tail
```
**Interpretation:** "Given that we fall into the worst 5% of scenarios, the average loss is `|cvar|`."

CVaR is a **better risk measure than VaR** because:
- VaR only tells you the *threshold* — not how bad it gets beyond that point
- CVaR tells you the *average severity* of tail losses
- CVaR is **coherent** (satisfies subadditivity — diversification always helps)

| Metric | Answers | Limitation |
|--------|---------|-----------|
| VaR 95% | "What's the worst loss in a normal year?" | Ignores the shape of the tail |
| CVaR 95% | "How bad is it when things go really wrong?" | Slightly harder to estimate accurately |


In [11]:
# PHASE 6: Risk Metrics Calculation
# Profit and Loss (P&L)
pnl = final_portfolio - INITIAL_PORTFOLIO

# Expected values
expected_final = np.mean(final_portfolio)
expected_return = (expected_final / INITIAL_PORTFOLIO) - 1

# Value at Risk (5th percentile of P&L)
var_95 = np.percentile(pnl, 5)

tail_mask = pnl <= var_95
cvar = pnl[tail_mask].mean()


## Output — Results & Performance Check

The final cell prints a formatted report and asserts the runtime is under 10 seconds.

### Reading the output
- **Expected Final Value** — average across all 50,000 simulations; should be above $1M given positive expected returns
- **Expected Annual Return** — `(expected_final / 1_000_000) - 1`; compare to the weighted average of `annual_returns` = 8.2% as a sanity check
- **VaR** — e.g. "$147,880" means there's a 5% chance of losing more than $147K in a year
- **CVaR** — e.g. "$194,526" means in the worst 5% of years, you lose on average $194K

### Why `{-var_95:,.2f}` (the negative sign)?
`var_95` from `np.percentile(pnl, 5)` is a *negative* number (it's a loss).  
We negate it to display a positive dollar figure: "you could lose $X".


In [13]:
# Output Results
print("\n" + "=" * 50)
print("MONTE CARLO PORTFOLIO RISK SIMULATION")
print("=" * 50)
print(f"Initial Investment:      ${INITIAL_PORTFOLIO:,.0f}")
print(f"Number of Simulations:   {N_SIMULATIONS:,}")
print(f"Time Horizon:            {N_DAYS} trading days (1 year)")
print("-" * 50)
print(f"Simulation Runtime:      {elapsed:.3f} seconds")
print("-" * 50)
print(f"Expected Final Value:    ${expected_final:,.2f}")
print(f"Expected Annual Return:  {expected_return * 100:.2f}%")
print("-" * 50)
print(f"5% Value at Risk (VaR):  ${-var_95:,.2f}")
print(f"   (5% chance of losing more than this amount)")
print(f"5% Conditional VaR:      ${-cvar:,.2f}")
print(f"   (Average loss in worst 5% of scenarios)")
print("=" * 50)

assert elapsed < 10.0, f"Simulation took {elapsed:.2f}s, should be <10s"
print("\nPerformance target met (under 10 second).")


MONTE CARLO PORTFOLIO RISK SIMULATION
Initial Investment:      $1,000,000
Number of Simulations:   50,000
Time Horizon:            252 trading days (1 year)
--------------------------------------------------
Simulation Runtime:      4.223 seconds
--------------------------------------------------
Expected Final Value:    $1,084,783.81
Expected Annual Return:  8.48%
--------------------------------------------------
5% Value at Risk (VaR):  $147,880.47
   (5% chance of losing more than this amount)
5% Conditional VaR:      $194,526.41
   (Average loss in worst 5% of scenarios)

Performance target met (under 10 second).
